In [1]:
from pathlib import Path
from typing import List
import pandas as pd
import sys
import yaml
PROJECT_ROOT = Path(".").resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))
from data_classes import timepoint, roi,neuron
from data_classes.video import VideoStatistics, VideoStatisticsWriter, Video
from utils.io_utils import load_model, load_config, save_node_level_comparisons
from pipeline.io_handlers import save_filtered_suite2p, visualize_neuron_groups
from utils.visualization import print_tree
from pipeline.video_runner import VideoPipelineRunner
from experiments.tree import ExperimentTreeBuilder, is_video_dir
from experiments.processor import ExperimentProcessor
from experiments.compare import ExperimentComparer, BasicSiblingComparator


In [2]:
config_path = PROJECT_ROOT / "config" / "notebook_config.yaml"
config = load_config(config_path)
spike_classifier = load_model(config["models"], which ="spike")
roi_classifier = load_model(config["models"], which ="roi")
models = {
    "spike": spike_classifier,
    "roi": roi_classifier
}
runner = VideoPipelineRunner.build(config)

In [3]:
builder = ExperimentTreeBuilder(is_video_dir=is_video_dir)
tree = builder.build(Path(r"C:\Users\mzinn1\Desktop\test_tree\ex345"))
print_tree(tree)    

└── ex345
    ├── AP5
    │   └── week 1
    │       ├── 2-1
    │       ├── 2-1_5um_1m
    │       ├── 2-2
    │       ├── 2-2_5um_3m
    │       ├── 2-3
    │       ├── 2-3_5um_5m
    │       └── metrics
    ├── GABA
    │   ├── metrics
    │   ├── week 1
    │   │   ├── 1-10_CTRL_2m
    │   │   ├── 1-7
    │   │   ├── 1-7_100nM_GABA_2m
    │   │   ├── 1-8
    │   │   └── metrics
    │   └── week 2
    │       ├── 1-10
    │       ├── 1-10_CTRL_2m
    │       ├── 1-9
    │       ├── 1-9_100nM_GABA_6m
    │       └── metrics
    └── metrics


In [4]:
experiment_root = r"C:\Users\mzinn1\Desktop\test_tree\ex345"
processor = ExperimentProcessor(runner=runner, models=models, config=config, output_root=experiment_root)
processor.process_tree(tree, verbose=False)

C:\Users\mzinn1\Desktop\Scripts\GCaMP-analysis\spike_processing\kinetics.py:42: RuntimeWarning: divide by zero encountered in scalar divide
  right_time = right_idx - (half_max - segment[right_idx]) / denom


In [5]:
comparer = ExperimentComparer(comparator=BasicSiblingComparator())
sibling_tables = comparer.compare_all(tree)
from experiments.io import save_node_level_comparisons_with_legend
save_node_level_comparisons_with_legend(
    root=tree,
    sibling_tables=sibling_tables,
    output_subdir="metrics",
    filename="sibling_comparisons.xlsx",
)
print("\n=== Sibling comparisons (by node) ===")
# Print the top-level node comparison if present
if experiment_root in sibling_tables:
    print(f"\nNode: {experiment_root}")
    print(sibling_tables[experiment_root].to_string(index=False))

# Print one level down comparisons too (often treatments)
for node_path, df in sibling_tables.items():
    if node_path == experiment_root:
        continue
    # keep output readable: only print “interesting” nodes
    if len(df) >= 2:
        print(f"\nNode: {node_path}")
        print(df.to_string(index=False))


=== Sibling comparisons (by node) ===

Node: C:\Users\mzinn1\Desktop\test_tree\ex345
                                 parent child  n_videos  n_neurons  decay_tau_mean_unweighted  decay_tau_var_unweighted  decay_tau_within_unweighted  decay_tau_between_unweighted  decay_tau_mean_weighted  decay_tau_var_weighted  decay_tau_within_weighted  decay_tau_between_weighted  half_max_width_mean_unweighted  half_max_width_var_unweighted  half_max_width_within_unweighted  half_max_width_between_unweighted  half_max_width_mean_weighted  half_max_width_var_weighted  half_max_width_within_weighted  half_max_width_between_weighted  rise_slope_mean_unweighted  rise_slope_var_unweighted  rise_slope_within_unweighted  rise_slope_between_unweighted  rise_slope_mean_weighted  rise_slope_var_weighted  rise_slope_within_weighted  rise_slope_between_weighted  spike_frequency_mean_unweighted  spike_frequency_var_unweighted  spike_frequency_within_unweighted  spike_frequency_between_unweighted  spike_frequenc